<a href="https://colab.research.google.com/github/gmatheushq/Analise-de-Credito/blob/main/Analise_de_Credito_IC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de Crédito: Exploração Inicial dos Dados

**Objetivo:** entender o perfil de pessoas que pedem empréstimo e quais características estão ligadas à **inadimplência** (não pagar o empréstimo).

**Dados:** `credit_risk_dataset.csv`, com 32.581 empréstimos e 12 colunas. Cada linha é um empréstimo, com dados da pessoa, do empréstimo e do histórico de crédito.


## 1. Conexão com o Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Bibliotecas e caminho do arquivo


In [31]:
import os
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ARQUIVO = "https://raw.githubusercontent.com/gmatheushq/Analise-de-Credito/main/credit_risk_dataset.csv"

print("Arquivo a ser lido:", ARQUIVO)

df = pd.read_csv(ARQUIVO)

print("Dimensões:", df.shape)
display(df.head())

Arquivo a ser lido: https://raw.githubusercontent.com/gmatheushq/Analise-de-Credito/main/credit_risk_dataset.csv
Dimensões: (32581, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## 3. Dicionário de dados

Cada linha do banco é um empréstimo.
### Colunas

| Coluna | Significado | Tipo |
|---|---|---|
| `person_age` | Idade da pessoa (anos) | Numérica |
| `person_income` | Renda anual da pessoa | Numérica |
| `person_home_ownership` | Tipo de propriedade da residência | Categórica |
| `person_emp_length` | Tempo de emprego (anos) | Numérica |
| `loan_intent` | Finalidade do empréstimo | Categórica |
| `loan_grade` | Classificação de risco do empréstimo | Categórica |
| `loan_amnt` | Valor solicitado do empréstimo | Numérica |
| `loan_int_rate` | Taxa de juros do empréstimo (%) | Numérica |
| `loan_status` | **Situação do empréstimo (variável-alvo)** | Binária |
| `loan_percent_income` | Percentual da renda comprometido com o empréstimo (0,15 = 15%) | Numérica |
| `cb_person_default_on_file` | Histórico de inadimplência registrado | Categórica |
| `cb_person_cred_hist_length` | Tempo de histórico de crédito (anos) | Numérica |

### Valores das variáveis categóricas

**`person_home_ownership`: tipo de propriedade da residência**

| Valor | Significado |
|---|---|
| `RENT` | Aluguel |
| `MORTGAGE` | Financiamento imobiliário |
| `OWN` | Imóvel próprio |
| `OTHER` | Outros |

**`loan_intent`: finalidade do empréstimo**

| Valor | Significado |
|---|---|
| `PERSONAL` | Pessoal |
| `EDUCATION` | Educação |
| `MEDICAL` | Médico |
| `VENTURE` | Empreendimento |
| `HOMEIMPROVEMENT` | Melhoria residencial |
| `DEBTCONSOLIDATION` | Consolidação de dívidas |

**`loan_grade`: classificação de risco do empréstimo**

| Valor | Significado |
|---|---|
| `A` | Risco muito baixo |
| `B` | Risco baixo |
| `C` | Risco moderado |
| `D` | Risco elevado |
| `E` | Risco muito elevado |
| `F` | Risco extremamente elevado |
| `G` | Risco máximo |

**`cb_person_default_on_file`: já teve inadimplência registrada?**

| Valor | Significado |
|---|---|
| `Y` | Sim |
| `N` | Não |

**`loan_status`: situação do empréstimo (variável-alvo)**

| Valor | Significado |
|---|---|
| `0` | Adimplente (pagou ou está em dia) |
| `1` | Inadimplente (não pagou) |

> `loan_status` é o que queremos entender e, mais adiante, prever.

## 3. Visualização do banco de dados


In [ ]:
display(Markdown("## Visualização das primeiras observações"))

df.head(5)

## Visualização das primeiras observações

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## 4. Carregamento e diagnóstico inicial
A próxima célula lê o CSV e gera um relatório em 5 partes:

1. **Estrutura:** número de linhas, colunas e células
2. **Variáveis:** tipo de cada coluna, valores ausentes e valores únicos
3. **Estatísticas descritivas:** média, mínimo, máximo e quartis
4. **Qualidade:** duplicados, ausentes e contagem de colunas numéricas e categóricas

O código é longo porque inclui formatação visual (cores e barras) para facilitar a leitura. A lógica em si é simples: `df.shape`, `df.head()`, `df.info()` e `df.describe()`.

In [ ]:


from IPython.display import display, Markdown

url = "https://raw.githubusercontent.com/gmatheushq/Analise-de-Credito/main/credit_risk_dataset.csv"

df = pd.read_csv(url)

# Configuração visual
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 40)


# 1. DIMENSÕES DO BANCO
display(Markdown("## 1. Estrutura do banco de dados"))

resumo = pd.DataFrame({
    "Indicador": ["Número de registros", "Número de variáveis", "Total de células"],
    "Valor": [
        f"{df.shape[0]:,}",
        f"{df.shape[1]:,}",
        f"{df.size:,}"
    ]
})

display(
    resumo.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#263445"),
                                      ("color", "white"),
                                      ("text-align", "center")]},
        {"selector": "td", "props": [("padding", "10px")]}
    ])
)


# 2. INFORMAÇÕES DAS VARIÁVEIS
display(Markdown("## 2. Informações gerais das variáveis"))

estrutura = pd.DataFrame({
    "Variável": df.columns,
    "Tipo": df.dtypes.astype(str).values,
    "Valores não nulos": df.notna().sum().values,
    "Valores ausentes": df.isna().sum().values,
    "Ausentes (%)": (df.isna().mean() * 100).values,
    "Valores únicos": df.nunique().values
})

display(
    estrutura.style
    .format({"Ausentes (%)": "{:.2f}%"})
    .background_gradient(subset=["Ausentes (%)"], cmap="Reds")
    .bar(subset=["Valores únicos"], color="#5B9BD5")
    .hide(axis="index")
)


# 3. ESTATÍSTICAS DESCRITIVAS
display(Markdown("## 3. Estatísticas descritivas"))

estatisticas = df.describe(include="all").T

display(
    estatisticas.style
    .format(precision=3, na_rep="—")
    .background_gradient(cmap="Blues", axis=0)
    .set_properties(**{"text-align": "center"})
)

# 4. RESUMO DA QUALIDADE DOS DADOS
display(Markdown("## 4. Diagnóstico inicial da qualidade dos dados"))

qualidade = pd.DataFrame({
    "Verificação": [
        "Registros duplicados",
        "Valores ausentes",
        "Variáveis com valores ausentes",
        "Variáveis numéricas",
        "Variáveis categóricas"
    ],
    "Resultado": [
        f"{df.duplicated().sum():,}",
        f"{df.isna().sum().sum():,}",
        f"{(df.isna().sum() > 0).sum():,}",
        f"{len(df.select_dtypes(include=np.number).columns)}",
        f"{len(df.select_dtypes(exclude=np.number).columns)}"
    ]
})

display(
    qualidade.style
    .hide(axis="index")
    .set_properties(**{"padding": "8px"})
)



## 1. Estrutura do banco de dados

Indicador,Valor
Número de registros,"32,581"
Número de variáveis,12
Total de células,"390,972"


## 2. Informações gerais das variáveis

Variável,Tipo,Valores não nulos,Valores ausentes,Ausentes (%),Valores únicos
person_age,int64,32581,0,0.00%,58
person_income,int64,32581,0,0.00%,4295
person_home_ownership,object,32581,0,0.00%,4
person_emp_length,float64,31686,895,2.75%,36
loan_intent,object,32581,0,0.00%,6
loan_grade,object,32581,0,0.00%,7
loan_amnt,int64,32581,0,0.00%,753
loan_int_rate,float64,29465,3116,9.56%,348
loan_status,int64,32581,0,0.00%,2
loan_percent_income,float64,32581,0,0.00%,77


## 3. Estatísticas descritivas

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
person_age,32581.000,—,—,—,27.735,6.348,20.000,23.000,26.000,30.000,144.000
person_income,32581.000,—,—,—,66074.848,61983.119,4000.000,38500.000,55000.000,79200.000,6000000.000
person_home_ownership,32581,4,RENT,16446,—,—,—,—,—,—,—
person_emp_length,31686.000,—,—,—,4.790,4.143,0.000,2.000,4.000,7.000,123.000
loan_intent,32581,6,EDUCATION,6453,—,—,—,—,—,—,—
loan_grade,32581,7,A,10777,—,—,—,—,—,—,—
loan_amnt,32581.000,—,—,—,9589.371,6322.087,500.000,5000.000,8000.000,12200.000,35000.000
loan_int_rate,29465.000,—,—,—,11.012,3.240,5.420,7.900,10.990,13.470,23.220
loan_status,32581.000,—,—,—,0.218,0.413,0.000,0.000,0.000,0.000,1.000
loan_percent_income,32581.000,—,—,—,0.170,0.107,0.000,0.090,0.150,0.230,0.830


## 4. Diagnóstico inicial da qualidade dos dados

Verificação,Resultado
Registros duplicados,165
Valores ausentes,"4,011"
Variáveis com valores ausentes,2
Variáveis numéricas,8
Variáveis categóricas,4


In [ ]:
from IPython.display import display, Markdown

# 1. VARIÁVEIS NUMÉRICAS
display(Markdown("## 1. Análise das variáveis numéricas"))

numericas = df.select_dtypes(include="number")

estatisticas = numericas.describe().T

estatisticas["mediana"] = numericas.median()
estatisticas["assimetria"] = numericas.skew()
estatisticas["curtose"] = numericas.kurt()

# Reorganizar colunas
estatisticas = estatisticas[
    ["count", "mean", "50%", "std", "min",
     "25%", "75%", "max", "assimetria", "curtose"]
]

estatisticas = estatisticas.rename(columns={
    "count": "Contagem",
    "mean": "Média",
    "50%": "Mediana",
    "std": "Desvio padrão",
    "min": "Mínimo",
    "25%": "Q1",
    "75%": "Q3",
    "max": "Máximo"
})

display(
    estatisticas.style
    .format("{:,.3f}")
    .background_gradient(cmap="Blues", subset=["Média", "Mediana", "Desvio padrão"])
    .set_properties(**{"text-align": "center"})
)

# 2. VARIÁVEIS CATEGÓRICAS
display(Markdown("## 2. Análise das variáveis categóricas"))

categoricas = df.select_dtypes(include=["object", "category", "bool"])

for coluna in categoricas.columns:

    frequencia = df[coluna].value_counts(dropna=False)

    tabela = pd.DataFrame({
        "Categoria": frequencia.index.astype(str),
        "Frequência": frequencia.values,
        "Percentual (%)": frequencia.values / len(df) * 100
    })

    display(Markdown(f"### Variável: `{coluna}`"))

    display(
        tabela.style
        .format({"Percentual (%)": "{:.2f}%"})
        .background_gradient(cmap="Blues", subset=["Frequência"])
        .bar(subset=["Percentual (%)"], color="#5B9BD5")
        .hide(axis="index")
    )


## 1. Análise das variáveis numéricas

,Contagem,Média,Mediana,Desvio padrão,Mínimo,Q1,Q3,Máximo,assimetria,curtose
person_age,"32,581.000",27.735,26.000,6.348,20.000,23.000,30.000,144.000,2.581,18.561
person_income,"32,581.000","66,074.848","55,000.000","61,983.119","4,000.000","38,500.000","79,200.000","6,000,000.000",32.865,"2,693.273"
person_emp_length,"31,686.000",4.790,4.000,4.143,0.000,2.000,7.000,123.000,2.614,43.722
loan_amnt,"32,581.000","9,589.371","8,000.000","6,322.087",500.000,"5,000.000","12,200.000","35,000.000",1.192,1.424
loan_int_rate,"29,465.000",11.012,10.990,3.240,5.420,7.900,13.470,23.220,0.209,-0.672
loan_status,"32,581.000",0.218,0.000,0.413,0.000,0.000,0.000,1.000,1.365,-0.137
loan_percent_income,"32,581.000",0.170,0.150,0.107,0.000,0.090,0.230,0.830,1.065,1.224
cb_person_cred_hist_length,"32,581.000",5.804,4.000,4.055,2.000,3.000,8.000,30.000,1.662,3.716


## 2. Análise das variáveis categóricas

### Variável: `person_home_ownership`

Categoria,Frequência,Percentual (%)
RENT,16446,50.48%
MORTGAGE,13444,41.26%
OWN,2584,7.93%
OTHER,107,0.33%


### Variável: `loan_intent`

Categoria,Frequência,Percentual (%)
EDUCATION,6453,19.81%
MEDICAL,6071,18.63%
VENTURE,5719,17.55%
PERSONAL,5521,16.95%
DEBTCONSOLIDATION,5212,16.00%
HOMEIMPROVEMENT,3605,11.06%


### Variável: `loan_grade`

Categoria,Frequência,Percentual (%)
A,10777,33.08%
B,10451,32.08%
C,6458,19.82%
D,3626,11.13%
E,964,2.96%
F,241,0.74%
G,64,0.20%


### Variável: `cb_person_default_on_file`

Categoria,Frequência,Percentual (%)
N,26836,82.37%
Y,5745,17.63%


## Relatorio parcial

| Achado | Dado | Por que importa |
|---|---|---|
| Classes desbalanceadas | ~22% dos empréstimos são inadimplentes (`loan_status = 1`) | Acurácia sozinha engana; usar recall, AUC e PR-AUC |
| Valores ausentes | `loan_int_rate` 9,56% e `person_emp_length` 2,75% | Precisam de tratamento (imputação) |
| Linhas duplicadas | 165 registros | Remover antes de modelar |
| Valores impossíveis | Idade até 144 anos, emprego até 123 anos, renda até 6 milhões | Provavelmente erros de digitação; investigar e tratar |
| Histórico de inadimplência | 17,6% já tiveram inadimplência registrada | Candidata forte a variável preditiva |


## 5. Limpeza dos dados

Com base no diagnóstico, vamos:
1. **Remover duplicados:** 165 linhas repetidas
2. **Remover valores impossíveis:** idade acima de 100 anos e tempo de emprego acima de 60 anos (erros de digitação)
3. **Investigar a renda:** os valores extremos serão avaliados, mas por ora mantidos
4. **Não imputar os ausentes agora:** `loan_int_rate` e `person_emp_length` serão preenchidos só na etapa de modelagem, usando apenas o conjunto de treino, para não vazar informação do teste.

In [29]:
df_limpo = df.copy()
n_inicial = len(df_limpo)

# 1. Duplicados
df_limpo = df_limpo.drop_duplicates()
n_dup = n_inicial - len(df_limpo)

# 2. Valores impossíveis (NaN não é removido aqui)
n_antes = len(df_limpo)
df_limpo = df_limpo[~(df_limpo["person_age"] > 100)]
n_idade = n_antes - len(df_limpo)

n_antes = len(df_limpo)
df_limpo = df_limpo[~(df_limpo["person_emp_length"] > 60)]
n_emprego = n_antes - len(df_limpo)

df_limpo = df_limpo.reset_index(drop=True)

# Resumo
resumo = pd.DataFrame({
    "Etapa": ["Registros iniciais", "Duplicados removidos",
              "Idade > 100 removidos", "Tempo de emprego > 60 removidos",
              "Registros finais"],
    "Quantidade": [n_inicial, n_dup, n_idade, n_emprego, len(df_limpo)]
})
display(resumo.style.hide(axis="index"))

print(f"\nTaxa de inadimplência antes: {df['loan_status'].mean():.2%}")
print(f"Taxa de inadimplência depois: {df_limpo['loan_status'].mean():.2%}")

# Máximos após a limpeza
print("\nMáximos após a limpeza:")
print(df_limpo[["person_age", "person_emp_length"]].max())

# Renda: olhar a cauda antes de decidir o que fazer
print("\nPercentis da renda:")
print(df_limpo["person_income"].quantile([.5, .9, .99, .999, 1]))

# Os ausentes têm relação com a inadimplência?
for col in ["loan_int_rate", "person_emp_length"]:
    print(f"\nInadimplência por ausência em {col}:")
    print(df_limpo.groupby(df_limpo[col].isna())["loan_status"].mean())

Etapa,Quantidade
Registros iniciais,32581
Duplicados removidos,165
Idade > 100 removidos,5
Tempo de emprego > 60 removidos,2
Registros finais,32409



Taxa de inadimplência antes: 21.82%
Taxa de inadimplência depois: 21.87%

Máximos após a limpeza:
person_age           94.0
person_emp_length    41.0
dtype: float64

Percentis da renda:
0.500      55000.0
0.900     110004.0
0.990     225000.0
0.999     643104.0
1.000    2039784.0
Name: person_income, dtype: float64

Inadimplência por ausência em loan_int_rate:
loan_int_rate
False    0.219922
True     0.207175
Name: loan_status, dtype: float64

Inadimplência por ausência em person_emp_length:
person_emp_length
False    0.215944
True     0.316798
Name: loan_status, dtype: float64
